# Colab + GitHub pipeline

Notebook base para trabajar el proyecto desde Colab Pro+ conectado a GitHub.

Este notebook está pensado para:
- clonar el repositorio
- montar Google Drive
- cargar datos
- reconstruir clusters
- construir adyacencias
- dejar listo el baseline GCN + LSTM


In [1]:
# 1. Clonar el repositorio
REPO_URL = "https://github.com/maisaarial/prediccion-congestion-trafico-gnn.git"
!rm -rf /content/prediccion-congestion-trafico-gnn
!git clone {REPO_URL}
%cd /content/prediccion-congestion-trafico-gnn


Cloning into 'prediccion-congestion-trafico-gnn'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 74 (delta 5), reused 74 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 33.90 KiB | 938.00 KiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/prediccion-congestion-trafico-gnn


In [2]:
!pwd
!ls

/content/prediccion-congestion-trafico-gnn
configs  data  notebooks  README.md  requirements.txt  results	src


In [3]:
# 2. Instalar dependencias
!pip install -q -r requirements.txt


In [4]:
# 3. Añadir src al path
import sys
from pathlib import Path

sys.path.append(str(Path('/content/prediccion-congestion-trafico-gnn/src')))


In [5]:
USUARIO = "ISABEL" # @param ["ISABEL","ENRIQUE","ASIER"]
import os
from google.colab import drive

rutas = {
    "ISABEL": "/content/drive/MyDrive/2026 Isabel Aristizabal/codigo/datos",
    "ENRIQUE": "/content/drive/MyDrive/2-ESTUDIANTES/2026 Isabel Aristizabal/codigo", # <--- Pon aquí la ruta tuya
    "ASIER": "/content/drive/MyDrive/INVESTIGACION/2026 Isabel Aristizabal/codigo"      # <--- Pon aquí la ruta tuya
}

DIRECTORIO = rutas[USUARIO]

# 4. Montar Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Configurar Git
!git config --global user.name "Maria Isabel Aristizabal"
!git config --global user.email "maisaarial@gmail.com"

In [7]:
# 5. Imports principales
import pandas as pd
import numpy as np
import torch
import os
import json
import matplotlib.pyplot as plt
from pathlib import Path

from torch.utils.data import DataLoader

from traffic_gnn.data.io import read_traffic_csv, read_sensors_csv
from traffic_gnn.features.congestion import calcular_congestion
from traffic_gnn.features.temporal import obtener_variables_temporales
from traffic_gnn.clustering.behavior import calcular_pivote_cl_comp, generar_cluster_comportamiento
from traffic_gnn.clustering.proximity import generar_cluster_proximidad
from traffic_gnn.clustering.intersections import intersectar_clusters
from traffic_gnn.graph.aggregation import aggregation_congestion_por_clusters, calcular_centroides_clusters
from traffic_gnn.graph.adjacency import adjacency_knn_from_centroids, build_cluster_time_matrix, adjacency_correlation_topk, to_pyg_tensors
from traffic_gnn.graph.datasets import crear_ventanas, split_temporal, build_tensor_datasets
from traffic_gnn.models.gcn_lstm import GCN_LSTM
from traffic_gnn.training.engine import train_one_epoch, evaluate


In [ ]:
# Crear los dos modelos nuevos dentro del repositorio
# Así quedan disponibles como archivos .py igual que gcn_lstm.py

from pathlib import Path

models_dir = Path("/content/prediccion-congestion-trafico-gnn/src/traffic_gnn/models")
models_dir.mkdir(parents=True, exist_ok=True)

gcn_gru_code = r"""
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv


def _batch_edge_index(edge_index, num_nodes, batch_size, device):
    edge_index = edge_index.to(device)
    offsets = torch.arange(batch_size, device=device).view(-1, 1, 1) * num_nodes
    edge_index_batched = edge_index.unsqueeze(0) + offsets
    edge_index_batched = edge_index_batched.permute(1, 0, 2).reshape(2, -1)
    return edge_index_batched


def _batch_edge_weight(edge_weight, batch_size, device):
    if edge_weight is None:
        return None
    return edge_weight.to(device).repeat(batch_size)


class GCN_GRU(nn.Module):
    def __init__(self, num_nodes, in_channels=1, gcn_hidden=32, gru_hidden=64):
        super().__init__()
        self.num_nodes = num_nodes
        self.gcn = GCNConv(in_channels, gcn_hidden)
        self.gru = nn.GRU(input_size=gcn_hidden, hidden_size=gru_hidden, batch_first=True)
        self.fc = nn.Linear(gru_hidden, 1)

    def forward(self, x, edge_index, edge_weight=None):
        # x puede llegar como (batch, T, N) o como (batch, T, N, F)
        if x.dim() == 3:
            x = x.unsqueeze(-1)

        # x: (batch, T, N, F)
        batch_size, T, N, F = x.shape
        device = x.device

        edge_index_batched = _batch_edge_index(edge_index, N, batch_size, device)
        edge_weight_batched = _batch_edge_weight(edge_weight, batch_size, device)

        gcn_outputs = []

        for t in range(T):
            x_t = x[:, t, :, :]                  # (batch, N, F)
            x_t = x_t.reshape(batch_size * N, F) # (batch*N, F)

            h_t = self.gcn(x_t, edge_index_batched, edge_weight_batched)
            h_t = torch.relu(h_t)
            h_t = h_t.reshape(batch_size, N, -1) # (batch, N, gcn_hidden)

            gcn_outputs.append(h_t)

        h = torch.stack(gcn_outputs, dim=1)      # (batch, T, N, gcn_hidden)
        h = h.permute(0, 2, 1, 3)                # (batch, N, T, gcn_hidden)
        h = h.reshape(batch_size * N, T, -1)     # (batch*N, T, gcn_hidden)

        h, _ = self.gru(h)                       # (batch*N, T, gru_hidden)
        h_last = h[:, -1, :]                     # (batch*N, gru_hidden)

        out = self.fc(h_last)                    # (batch*N, 1)
        out = out.reshape(batch_size, N)         # (batch, N)

        return out
"""

gat_lstm_code = r"""
import torch
import torch.nn as nn
from torch_geometric.nn import GATConv


def _batch_edge_index(edge_index, num_nodes, batch_size, device):
    edge_index = edge_index.to(device)
    offsets = torch.arange(batch_size, device=device).view(-1, 1, 1) * num_nodes
    edge_index_batched = edge_index.unsqueeze(0) + offsets
    edge_index_batched = edge_index_batched.permute(1, 0, 2).reshape(2, -1)
    return edge_index_batched


class GAT_LSTM(nn.Module):
    def __init__(self, num_nodes, in_channels=1, gat_hidden=32, lstm_hidden=64, heads=1):
        super().__init__()
        self.num_nodes = num_nodes
        self.gat = GATConv(
            in_channels=in_channels,
            out_channels=gat_hidden,
            heads=heads,
            concat=False
        )
        self.lstm = nn.LSTM(input_size=gat_hidden, hidden_size=lstm_hidden, batch_first=True)
        self.fc = nn.Linear(lstm_hidden, 1)

    def forward(self, x, edge_index, edge_weight=None):
        # x puede llegar como (batch, T, N) o como (batch, T, N, F)
        if x.dim() == 3:
            x = x.unsqueeze(-1)

        # x: (batch, T, N, F)
        # edge_weight se recibe para mantener compatibilidad con el engine,
        # pero GAT aprende los pesos de atención y no lo usa directamente.
        batch_size, T, N, F = x.shape
        device = x.device

        edge_index_batched = _batch_edge_index(edge_index, N, batch_size, device)

        gat_outputs = []

        for t in range(T):
            x_t = x[:, t, :, :]                  # (batch, N, F)
            x_t = x_t.reshape(batch_size * N, F) # (batch*N, F)

            h_t = self.gat(x_t, edge_index_batched)
            h_t = torch.relu(h_t)
            h_t = h_t.reshape(batch_size, N, -1) # (batch, N, gat_hidden)

            gat_outputs.append(h_t)

        h = torch.stack(gat_outputs, dim=1)      # (batch, T, N, gat_hidden)
        h = h.permute(0, 2, 1, 3)                # (batch, N, T, gat_hidden)
        h = h.reshape(batch_size * N, T, -1)     # (batch*N, T, gat_hidden)

        h, _ = self.lstm(h)                      # (batch*N, T, lstm_hidden)
        h_last = h[:, -1, :]                     # (batch*N, lstm_hidden)

        out = self.fc(h_last)                    # (batch*N, 1)
        out = out.reshape(batch_size, N)         # (batch, N)

        return out
"""

(models_dir / "gcn_gru.py").write_text(gcn_gru_code, encoding="utf-8")
(models_dir / "gat_lstm.py").write_text(gat_lstm_code, encoding="utf-8")

print("Modelos creados:")
print(models_dir / "gcn_gru.py")
print(models_dir / "gat_lstm.py")

# Importar modelos nuevos
import importlib
importlib.invalidate_caches()

from traffic_gnn.models.gcn_gru import GCN_GRU
from traffic_gnn.models.gat_lstm import GAT_LSTM

print("Imports OK: GCN_GRU y GAT_LSTM")

## 6. Configurar las rutas de los datos


In [8]:
BASE_PATH = "/content/drive/MyDrive/2026 Isabel Aristizabal/codigo/datos"

TRAFFIC_PATH = f"{BASE_PATH}/madrid_historico_datos_trafico/2025/01-2025.csv"
SENSORS_PATH = f"{BASE_PATH}/madrid_historico_puntos_medida/2025/pmed_ubicacion_01-2025.csv"


In [9]:
cols_trafico = ['id', 'fecha', 'tipo_elem', 'intensidad', 'ocupacion']
cols_puntos = ['id', 'distrito', 'nombre', 'utm_x', 'utm_y', 'longitud', 'latitud']

trafico_raw = read_traffic_csv(TRAFFIC_PATH, usecols=cols_trafico)
sensores_raw = read_sensors_csv(SENSORS_PATH, usecols=cols_puntos)

print(trafico_raw.shape)
print(sensores_raw.shape)


(13218804, 5)
(4960, 7)


## 7. Preparación base

In [10]:
trafico = calcular_congestion(trafico_raw)
trafico = obtener_variables_temporales(trafico)

pivote_cl_comp = calcular_pivote_cl_comp(trafico)
pivote_cl_comp = pivote_cl_comp.dropna()
lista_ids_validos = pivote_cl_comp.index

sensores = sensores_raw[sensores_raw['id'].isin(lista_ids_validos)].copy()
trafico = trafico[trafico['id'].isin(lista_ids_validos)].copy()

print('Sensores válidos:', sensores['id'].nunique())


Sensores válidos: 4603


## 8. Caso: proximidad

In [11]:
EPS_PROX_500 = 171.63
cluster_proximidad_500 = generar_cluster_proximidad(sensores, epsilon=EPS_PROX_500)
cluster_proximidad_500 = cluster_proximidad_500[['id', 'nombre', 'cluster_proximidad']]
cluster_proximidad_500.head()


,id,nombre,cluster_proximidad
0,3937,Serrano N-S - General Oraa-Diego de Leon,0
1,5947,Hermanos García Noblejas - Dr. Cirajas-Vital Aza,1
2,10307,Santa Genoveva - Santa Genoveva-Av. Trece Rosas,1
3,6135,Av. Marqués de Corbera - Ntra. Sra. del Villar...,1
4,10447,Francisco Villaespesa - José María Pereda-Herm...,1


## 9. Caso: proximidad ∩ comportamiento

In [12]:
pivote_clusterizado = generar_cluster_comportamiento(pivote_cl_comp, n_clusters=2)
cluster_comportamiento = pivote_clusterizado.reset_index()[['id', 'cluster_comportamiento']]

cluster_prox_tmp = generar_cluster_proximidad(sensores, epsilon=195.43)[['id', 'nombre', 'cluster_proximidad']]
cluster_prox_comp_500 = intersectar_clusters(cluster_prox_tmp, cluster_comportamiento)
cluster_prox_comp_500.head()


/content/prediccion-congestion-trafico-gnn/src/traffic_gnn/clustering/intersections.py:16: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  df[output_col] = pd.factorize(pares)[0]


,id,nombre,cluster_proximidad,cluster_comportamiento,cluster_interseccion
0,3937,Serrano N-S - General Oraa-Diego de Leon,0,0,0
1,5947,Hermanos García Noblejas - Dr. Cirajas-Vital Aza,0,0,0
2,10307,Santa Genoveva - Santa Genoveva-Av. Trece Rosas,0,0,0
3,6135,Av. Marqués de Corbera - Ntra. Sra. del Villar...,0,0,0
4,10447,Francisco Villaespesa - José María Pereda-Herm...,0,0,0


## 10. Agregación por cluster

In [13]:
df_merge = trafico.merge(cluster_proximidad_500[['id', 'cluster_proximidad']], on='id', how='left')
agg = aggregation_congestion_por_clusters(df_merge, nombre_col_cluster='cluster_proximidad')
agg.head()


,cluster,fecha,congestion
0,0,2025-01-01 00:00:00,0.025677
1,0,2025-01-01 00:15:00,0.013882
2,0,2025-01-01 00:30:00,0.023129
3,0,2025-01-01 00:45:00,0.020123
4,0,2025-01-01 01:00:00,0.019738


## 11. Adyacencia por cercanía

In [14]:
info_centroides = cluster_proximidad_500.merge(sensores[['id', 'utm_x', 'utm_y']], on='id', how='left')
centroides = calcular_centroides_clusters(info_centroides, cluster_col='cluster_proximidad')
edges_knn = adjacency_knn_from_centroids(centroides, k=5)
edges_knn.head()


,source,target,distance,weight
0,0,362,286.854249,0.644321
1,0,465,709.924756,0.336942
2,0,396,829.958376,0.280333
3,0,312,886.637226,0.257013
4,0,348,903.201545,0.250572


## 12. Adyacencia por correlación

In [15]:
tabla = build_cluster_time_matrix(agg)
tabla_gnn, edges_corr, mapping, mapping_inverso = adjacency_correlation_topk(tabla, k=5)
edge_index, edge_weight = to_pyg_tensors(edges_corr)

print(tabla_gnn.shape)
print(edge_index.shape)
print(edge_weight.shape)


(2965, 495)
torch.Size([2, 2475])
torch.Size([2475])


## 13. Ventanas temporales

In [16]:
data = tabla_gnn.values.astype(np.float32)
X, y = crear_ventanas(data, window=12, horizon=1)
X_train, y_train, X_val, y_val, X_test, y_test = split_temporal(X, y)

train_dataset, val_dataset, test_dataset = build_tensor_datasets(X_train, y_train, X_val, y_val, X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(X_train.shape, y_train.shape)


(2067, 12, 495, 1) (2067, 495)


## 14. Modelo GCN + LSTM

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN_LSTM(num_nodes=X_train.shape[2], in_channels=1, gcn_hidden=32, lstm_hidden=64).to(device)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
edge_index = edge_index.to(device)
edge_weight = edge_weight.to(device)
print(device)


cpu


## 15. Entrenamiento rápido de prueba

In [18]:
epochs = 3
best_val_loss = float('inf')
best_state = None

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, edge_index, edge_weight, device)
    val_loss, val_mae, val_rmse, _, _ = evaluate(model, val_loader, criterion, edge_index, edge_weight, device)
    print(f'Epoch {epoch+1:02d} | Train {train_loss:.6f} | Val {val_loss:.6f} | MAE {val_mae:.6f} | RMSE {val_rmse:.6f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = model.state_dict()


Epoch 01 | Train 0.003869 | Val 0.002834 | MAE 0.024847 | RMSE 0.053234
Epoch 02 | Train 0.002746 | Val 0.002751 | MAE 0.024160 | RMSE 0.052454
Epoch 03 | Train 0.002706 | Val 0.002718 | MAE 0.023523 | RMSE 0.052132


## 16. Evaluación

In [50]:
model.load_state_dict(best_state)
test_loss, test_mae, test_rmse, y_pred_test, y_true_test = evaluate(model, test_loader, criterion, edge_index, edge_weight, device)
print('Test loss:', test_loss)
print('Test MAE :', test_mae)
print('Test RMSE:', test_rmse)


Test loss: 0.0026748419653111334
Test MAE : 0.02333931438624859
Test RMSE: 0.05171887204051018


## 17. Pruebas completas: GCN+LSTM, GCN+GRU y GAT+LSTM

Este bloque ejecuta los tres modelos en todos los casos definidos, guarda resultados, modelos y gráficas, y además muestra las comparaciones en pantalla.

In [ ]:
# ============================================================
# CONFIGURACIÓN DE EXPERIMENTOS
# ============================================================

RESULTS_DIR = Path("/content/prediccion-congestion-trafico-gnn/resultados/experimentos_gnn")
FIGURES_DIR = RESULTS_DIR / "figuras"
MODELS_OUT_DIR = RESULTS_DIR / "modelos"
PRED_DIR = RESULTS_DIR / "predicciones"

for d in [RESULTS_DIR, FIGURES_DIR, MODELS_OUT_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EPOCHS_EXPERIMENTOS = 3
BATCH_SIZE_EXPERIMENTOS = 16
LR_EXPERIMENTOS = 1e-3
K_VECINOS = 5
WINDOW = 12
HORIZON = 1

print("Carpeta de resultados:", RESULTS_DIR)

In [ ]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def detectar_columna_cluster(df_cluster):
    """
    Detecta automáticamente la columna de cluster.
    Sirve para casos como:
    - cluster_proximidad
    - cluster_interseccion
    - cluster_prox_comp
    """
    candidatas = [
        c for c in df_cluster.columns
        if "cluster" in c.lower() and c.lower() not in ["cluster_comportamiento"]
    ]

    # Priorizamos columnas de intersección si existen
    candidatas_inter = [
        c for c in candidatas
        if "inter" in c.lower() or "prox_comp" in c.lower() or "proximidad_comportamiento" in c.lower()
    ]

    if candidatas_inter:
        return candidatas_inter[-1]

    if candidatas:
        return candidatas[-1]

    raise ValueError("No se encontró ninguna columna de cluster en el dataframe.")


def preparar_caso_gnn(nombre_caso, df_cluster, cluster_col=None, k=5, window=12, horizon=1):
    """
    Construye:
    - agregación temporal por cluster
    - adyacencia por cercanía
    - adyacencia por correlación
    - ventanas temporales
    - dataloaders
    """

    if cluster_col is None:
        cluster_col = detectar_columna_cluster(df_cluster)

    print(f"\nPreparando caso: {nombre_caso}")
    print(f"Columna de cluster usada: {cluster_col}")

    df_cluster_tmp = df_cluster[["id", cluster_col]].copy()
    df_cluster_tmp = df_cluster_tmp.rename(columns={cluster_col: "cluster_experimento"})

    # Agregación por cluster
    df_merge = trafico.merge(df_cluster_tmp, on="id", how="left")
    df_merge = df_merge.dropna(subset=["cluster_experimento"]).copy()

    agg_caso = aggregation_congestion_por_clusters(
        df_merge,
        nombre_col_cluster="cluster_experimento"
    )

    # Matriz temporal cluster x tiempo
    tabla = build_cluster_time_matrix(agg_caso)

    # Adyacencia por correlación
    tabla_corr, edges_corr, mapping_corr, mapping_inverso_corr = adjacency_correlation_topk(tabla, k=k)
    edge_index_corr, edge_weight_corr = to_pyg_tensors(edges_corr)

    # Adyacencia por cercanía
    info_centroides = df_cluster_tmp.merge(
        sensores[["id", "utm_x", "utm_y"]],
        on="id",
        how="left"
    )

    centroides = calcular_centroides_clusters(
        info_centroides,
        cluster_col="cluster_experimento"
    )

    edges_knn = adjacency_knn_from_centroids(centroides, k=k)

    # Para mantener las mismas columnas/nodos que tabla_corr:
    # usamos tabla_corr como matriz de datos y la adyacencia KNN convertida a PyG.
    edge_index_knn, edge_weight_knn = to_pyg_tensors(edges_knn)

    casos_adyacencia = {
        "correlacion": {
            "tabla_gnn": tabla_corr,
            "edge_index": edge_index_corr,
            "edge_weight": edge_weight_corr
        },
        "cercania_knn": {
            "tabla_gnn": tabla_corr,
            "edge_index": edge_index_knn,
            "edge_weight": edge_weight_knn
        }
    }

    salidas = {}

    for nombre_adj, info_adj in casos_adyacencia.items():
        tabla_gnn = info_adj["tabla_gnn"]
        data = tabla_gnn.values.astype(np.float32)

        X, y = crear_ventanas(data, window=window, horizon=horizon)
        X_train, y_train, X_val, y_val, X_test, y_test = split_temporal(X, y)

        train_dataset, val_dataset, test_dataset = build_tensor_datasets(
            X_train, y_train, X_val, y_val, X_test, y_test
        )

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_EXPERIMENTOS, shuffle=False)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_EXPERIMENTOS, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE_EXPERIMENTOS, shuffle=False)

        salidas[nombre_adj] = {
            "tabla_gnn": tabla_gnn,
            "X_train": X_train,
            "y_train": y_train,
            "X_val": X_val,
            "y_val": y_val,
            "X_test": X_test,
            "y_test": y_test,
            "train_loader": train_loader,
            "val_loader": val_loader,
            "test_loader": test_loader,
            "edge_index": info_adj["edge_index"],
            "edge_weight": info_adj["edge_weight"],
            "num_nodes": X_train.shape[2],
            "in_channels": 1 if len(X_train.shape) == 3 else X_train.shape[3],
        }

        print(
            f"  {nombre_adj}: X_train={X_train.shape}, "
            f"edge_index={info_adj['edge_index'].shape}"
        )

    return salidas


def crear_modelo(nombre_modelo, num_nodes, in_channels):
    if nombre_modelo == "GCN_LSTM":
        return GCN_LSTM(
            num_nodes=num_nodes,
            in_channels=in_channels,
            gcn_hidden=32,
            lstm_hidden=64
        )

    if nombre_modelo == "GCN_GRU":
        return GCN_GRU(
            num_nodes=num_nodes,
            in_channels=in_channels,
            gcn_hidden=32,
            gru_hidden=64
        )

    if nombre_modelo == "GAT_LSTM":
        return GAT_LSTM(
            num_nodes=num_nodes,
            in_channels=in_channels,
            gat_hidden=32,
            lstm_hidden=64,
            heads=1
        )

    raise ValueError(f"Modelo no reconocido: {nombre_modelo}")


def entrenar_y_evaluar_modelo(nombre_modelo, datos, nombre_caso, nombre_adj, epochs=3):
    model = crear_modelo(
        nombre_modelo=nombre_modelo,
        num_nodes=datos["num_nodes"],
        in_channels=datos["in_channels"]
    ).to(device)

    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_EXPERIMENTOS)

    edge_index_exp = datos["edge_index"].to(device)
    edge_weight_exp = datos["edge_weight"].to(device)

    history = []
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(epochs):
        train_loss = train_one_epoch(
            model,
            datos["train_loader"],
            optimizer,
            criterion,
            edge_index_exp,
            edge_weight_exp,
            device
        )

        val_loss, val_mae, val_rmse, _, _ = evaluate(
            model,
            datos["val_loader"],
            criterion,
            edge_index_exp,
            edge_weight_exp,
            device
        )

        history.append({
            "caso": nombre_caso,
            "adyacencia": nombre_adj,
            "modelo": nombre_modelo,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_mae": val_mae,
            "val_rmse": val_rmse
        })

        print(
            f"{nombre_caso} | {nombre_adj} | {nombre_modelo} | "
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"Train {train_loss:.6f} | Val {val_loss:.6f} | "
            f"MAE {val_mae:.6f} | RMSE {val_rmse:.6f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

    model.load_state_dict(best_state)

    test_loss, test_mae, test_rmse, y_pred_test, y_true_test = evaluate(
        model,
        datos["test_loader"],
        criterion,
        edge_index_exp,
        edge_weight_exp,
        device
    )

    resultado = {
        "caso": nombre_caso,
        "adyacencia": nombre_adj,
        "modelo": nombre_modelo,
        "test_loss_mse": test_loss,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "best_val_loss": best_val_loss,
        "epochs": epochs,
        "num_nodes": datos["num_nodes"],
        "in_channels": datos["in_channels"]
    }

    nombre_base = f"{nombre_caso}__{nombre_adj}__{nombre_modelo}"

    # Guardar modelo
    torch.save(
        model.state_dict(),
        MODELS_OUT_DIR / f"{nombre_base}.pt"
    )

    # Guardar predicciones
    np.save(PRED_DIR / f"{nombre_base}_y_pred.npy", y_pred_test)
    np.save(PRED_DIR / f"{nombre_base}_y_true.npy", y_true_test)

    return resultado, pd.DataFrame(history), y_pred_test, y_true_test


def graficar_historial(df_hist, nombre_caso, nombre_adj):
    plt.figure(figsize=(10, 5))

    for modelo_nombre in df_hist["modelo"].unique():
        df_m = df_hist[df_hist["modelo"] == modelo_nombre]
        plt.plot(df_m["epoch"], df_m["train_loss"], marker="o", label=f"{modelo_nombre} train")
        plt.plot(df_m["epoch"], df_m["val_loss"], marker="o", linestyle="--", label=f"{modelo_nombre} val")

    plt.title(f"Evolución de pérdidas - {nombre_caso} - {nombre_adj}")
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    path = FIGURES_DIR / f"loss_{nombre_caso}_{nombre_adj}.png"
    plt.savefig(path, dpi=150)
    plt.show()

    print("Gráfica guardada en:", path)


def graficar_comparacion_metricas(df_resultados):
    metricas = ["test_mae", "test_rmse", "test_loss_mse"]

    for metrica in metricas:
        plt.figure(figsize=(12, 5))

        etiquetas = (
            df_resultados["caso"] + "\n" +
            df_resultados["adyacencia"] + "\n" +
            df_resultados["modelo"]
        )

        plt.bar(etiquetas, df_resultados[metrica])
        plt.title(f"Comparación de modelos - {metrica}")
        plt.ylabel(metrica)
        plt.xticks(rotation=90)
        plt.tight_layout()

        path = FIGURES_DIR / f"comparacion_{metrica}.png"
        plt.savefig(path, dpi=150)
        plt.show()

        print("Gráfica guardada en:", path)


def graficar_pred_vs_real(y_true, y_pred, nombre_base, max_puntos=300):
    y_true_flat = np.array(y_true).reshape(-1)
    y_pred_flat = np.array(y_pred).reshape(-1)

    n = min(max_puntos, len(y_true_flat))

    plt.figure(figsize=(12, 5))
    plt.plot(y_true_flat[:n], label="Real")
    plt.plot(y_pred_flat[:n], label="Predicción")
    plt.title(f"Predicción vs real - {nombre_base}")
    plt.xlabel("Observaciones")
    plt.ylabel("Congestión")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    path = FIGURES_DIR / f"pred_vs_real_{nombre_base}.png"
    plt.savefig(path, dpi=150)
    plt.show()

    print("Gráfica guardada en:", path)

In [ ]:
# ============================================================
# DEFINICIÓN DE CASOS
# ============================================================

casos_clusters = {
    "proximidad_500": {
        "df_cluster": cluster_proximidad_500,
        "cluster_col": "cluster_proximidad"
    },
    "proximidad_comportamiento_500": {
        "df_cluster": cluster_prox_comp_500,
        "cluster_col": None  # se detecta automáticamente
    }
}

modelos_a_probar = [
    "GCN_LSTM",
    "GCN_GRU",
    "GAT_LSTM"
]

todos_resultados = []
todos_historiales = []

predicciones_guardadas = {}

for nombre_caso, info_caso in casos_clusters.items():

    datos_por_adyacencia = preparar_caso_gnn(
        nombre_caso=nombre_caso,
        df_cluster=info_caso["df_cluster"],
        cluster_col=info_caso["cluster_col"],
        k=K_VECINOS,
        window=WINDOW,
        horizon=HORIZON
    )

    for nombre_adj, datos in datos_por_adyacencia.items():

        historiales_adj = []

        for nombre_modelo in modelos_a_probar:
            resultado, df_hist, y_pred, y_true = entrenar_y_evaluar_modelo(
                nombre_modelo=nombre_modelo,
                datos=datos,
                nombre_caso=nombre_caso,
                nombre_adj=nombre_adj,
                epochs=EPOCHS_EXPERIMENTOS
            )

            todos_resultados.append(resultado)
            todos_historiales.append(df_hist)
            historiales_adj.append(df_hist)

            nombre_base = f"{nombre_caso}__{nombre_adj}__{nombre_modelo}"
            predicciones_guardadas[nombre_base] = {
                "y_pred": y_pred,
                "y_true": y_true
            }

            graficar_pred_vs_real(
                y_true=y_true,
                y_pred=y_pred,
                nombre_base=nombre_base,
                max_puntos=300
            )

        df_hist_adj = pd.concat(historiales_adj, ignore_index=True)
        graficar_historial(df_hist_adj, nombre_caso, nombre_adj)

df_resultados = pd.DataFrame(todos_resultados)
df_historial = pd.concat(todos_historiales, ignore_index=True)

# Guardar resultados tabulares
resultados_csv = RESULTS_DIR / "comparacion_resultados_modelos.csv"
historial_csv = RESULTS_DIR / "historial_entrenamiento_modelos.csv"

df_resultados.to_csv(resultados_csv, index=False)
df_historial.to_csv(historial_csv, index=False)

print("\nResultados guardados en:")
print(resultados_csv)
print(historial_csv)

df_resultados

In [ ]:
# ============================================================
# COMPARACIONES FINALES
# ============================================================

df_resultados_ordenado = df_resultados.sort_values("test_mae").reset_index(drop=True)

print("Ranking general por menor MAE:")
display(df_resultados_ordenado)

print("\nMejor modelo por cada caso y tipo de adyacencia:")
mejores_por_caso = (
    df_resultados
    .sort_values("test_mae")
    .groupby(["caso", "adyacencia"], as_index=False)
    .first()
)

display(mejores_por_caso)

# Guardar ranking y mejores modelos
ranking_csv = RESULTS_DIR / "ranking_general_modelos.csv"
mejores_csv = RESULTS_DIR / "mejores_modelos_por_caso.csv"

df_resultados_ordenado.to_csv(ranking_csv, index=False)
mejores_por_caso.to_csv(mejores_csv, index=False)

print("Ranking guardado en:", ranking_csv)
print("Mejores por caso guardado en:", mejores_csv)

# Gráficas comparativas
graficar_comparacion_metricas(df_resultados_ordenado)

# Subir cambios a Github

In [51]:
import os
import shutil
import subprocess
from getpass import getpass

NOTEBOOK_NAME = "01_colab_github_pipeline.ipynb"
NOTEBOOK_PATH_DRIVE = f"/content/drive/MyDrive/2026 Isabel Aristizabal/codigo/github/{NOTEBOOK_NAME}"
NOTEBOOK_PATH_REPO = f"/content/prediccion-congestion-trafico-gnn/notebooks/{NOTEBOOK_NAME}"

In [52]:
# Verificar que exista en Drive
if not os.path.exists(NOTEBOOK_PATH_DRIVE):
    raise FileNotFoundError(f"No existe el notebook en Drive: {NOTEBOOK_PATH_DRIVE}")

In [53]:
# Copiar notebook al repo
shutil.copy2(NOTEBOOK_PATH_DRIVE, NOTEBOOK_PATH_REPO)
print(f"Notebook copiado al repo:\n{NOTEBOOK_PATH_REPO}")

Notebook copiado al repo:
/content/prediccion-congestion-trafico-gnn/notebooks/01_colab_github_pipeline.ipynb


In [54]:
# Ir al repo
os.chdir("/content/prediccion-congestion-trafico-gnn")

In [55]:
# Configurar git
subprocess.run(["git", "config", "--global", "user.name", "Maria Isabel Aristizabal"], check=False)
subprocess.run(["git", "config", "--global", "user.email", "maisaarial@gmail.com"], check=False)


CompletedProcess(args=['git', 'config', '--global', 'user.email', 'maisaarial@gmail.com'], returncode=0)

In [56]:
!git branch
!git status
!git remote -v

* main
On branch main
Your branch is ahead of 'origin/main' by 2 commits.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/01_colab_github_pipeline.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
origin	https://github.com/maisaarial/prediccion-congestion-trafico-gnn.git (fetch)
origin	https://github.com/maisaarial/prediccion-congestion-trafico-gnn.git (push)


In [57]:
# Añadir cambios
subprocess.run(["git", "add", "."], check=True)

# Ver si hay cambios
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, check=True)

if not status.stdout.strip():
    print("No hay cambios para commit.")
else:
  token = getpass("Pega tu token NUEVO de GitHub: ").strip()
  repo_url = f"https://{token}@github.com/maisaarial/prediccion-congestion-trafico-gnn.git"

  subprocess.run(["git", "remote", "set-url", "origin", repo_url], check=True)
  subprocess.run(["git", "push", "origin", "main"], check=True)

Pega tu token NUEVO de GitHub: ··········


CalledProcessError: Command '['git', 'push', 'origin', 'main']' returned non-zero exit status 128.

In [46]:
!git remote set-url origin https://github.com/maisaarial/prediccion-congestion-trafico-gnn.git
!git remote -v

origin	https://github.com/maisaarial/prediccion-congestion-trafico-gnn.git (fetch)
origin	https://github.com/maisaarial/prediccion-congestion-trafico-gnn.git (push)
